# 09d: Cardio-70k, certificates, release, policy certification and results

Design decisions
- Everything comes from the frozen configuration of notebook 09c (hash in results/09c_frozen.json): configuration, model, prior, split and explained-subsample hashes are checked before any calibration row is read.
- As notebooks 08 and 09: candidate sets for the explained patients of calibration split 2 and of the test split from their features only; then certificates on calibration split 1; every box that could be released at the loosest target of either grid (0.30 for negative-outcome claims, 0.80 for enrichment claims) is re-checked by the second exact check, with Z3 on a random sample.
- Release, policy certification on calibration split 2 at the frozen targets (alpha_0 = 0.20, beta = 0.80, alpha_sel from 09c), test results on the 5,000 explained test patients, the consistency check inside released boxes on the full test split, and the model metrics.
- Rows for both grids: the scaled grid (alpha_0 0.10, 0.20, 0.30; beta 0.6, 0.7, 0.8) and the plan's grid (alpha_0 0.02, 0.05, 0.10; beta 0.2, 0.3, 0.5); only the frozen targets carry the policy guarantee.
- Candidate sets are saved on Drive every 500 new cells, so an interrupted run resumes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask, pair_arrays, release_patients, kind_summary, policy_test
from scax.explain.candidates import CandidateGenerator
from scax.explain.verify import verify_boxes, z3_sample
from scax.certify.counting import box_counts
from scax.certify.certificate import upper_bounds
from scax.certify.clopper_pearson import cp_lower, cp_upper
from scax.metrics.model_metrics import model_metrics

SEED = 42
seed_everything(SEED)
TASK = 'cardio70k'
LIMIT = None          # every frozen explained patient
REC = json.load(open(f'{paths.RESULTS}/09c_frozen.json'))
assert REC['written_before_calibration_split_1']
ft = FrozenTask(TASK, REPO_DIR, DRIVE_ROOT, REC['frozen_config_sha256'][TASK])
T = ft.targets
CERT_DIR = paths.DERIVED / 'certificates'; CERT_DIR.mkdir(parents=True, exist_ok=True)
X, y = public.LOADERS[TASK](paths.RAW)
Xa = X.to_numpy(dtype=float)
code = np.load(paths.DERIVED / 'splits' / f'{TASK}.npy')
assert splits.summary(code)['sha256'] == ft.cfg['split_sha256'], 'split differs from the frozen one'
cal2_rows = ft.cal2_rows if LIMIT is None else ft.cal2_rows[:LIMIT]
test_rows = ft.test_rows if LIMIT is None else ft.test_rows[:LIMIT]
GRIDS = {'scaled': (T['alpha_0_grid'], T['beta_grid']), 'plan': (T['plan_alpha_0_grid'], T['plan_beta_grid'])}
LOOSEST = {0: max(T['alpha_0_grid'] + T['plan_alpha_0_grid']), 1: round(1 - min(T['beta_grid'] + T['plan_beta_grid']), 10)}
print('frozen configuration', ft.cfg_sha[:16], '| model', ft.cfg['model']['sha256'][:16], ft.ens.input_cast,
      '| targets', {k: T[k] for k in ('alpha_0', 'beta', 'alpha_sel')}, '| explained', len(cal2_rows), len(test_rows))

## Candidate sets for the explained patients (features only)

In [ ]:
rows = np.r_[cal2_rows, test_rows]
C = ft.grid.cells(Xa[rows]); yhat = ft.pred.predict_cells(C).astype(int)
c = ft.cfg['candidates']
gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
assert gen.settings['inflation'] == c['inflation']
cache = CERT_DIR / f'{TASK}__candidates.pkl'
if gen.load(cache):
    print(f'{len(gen.cache)} cells loaded from Drive')
t0 = time.time()
for i in range(len(C)):
    if gen.cell_key(C[i]) not in gen.cache:
        gen.candidates(C[i], int(yhat[i]))
        if gen.oracle.stats.cells % 500 == 0:
            gen.save(cache)
            print(f'  {gen.oracle.stats.cells} new cells, {time.time() - t0:.0f}s', flush=True)
gen.save(cache)
gen.oracle.memo.clear()
print(f'{len(rows)} explained patients, {len(gen.cache)} distinct cells, median {np.median(list(gen.seconds.values())):.2f}s per cell')

## Certificates on calibration split 1 and the second exact check

In [ ]:
cal1 = splits.indices(code, 'cal1')
C1, y1 = ft.grid.cells(Xa[cal1]), y[cal1]
boxes = sorted({(cd.key, cd.y) for k in {gen.cell_key(r) for r in C} for cd in gen.cache[k]})
keys = [k for k, _ in boxes]; cls = np.array([cl for _, cl in boxes])
n_B, k_B = box_counts(keys, cls, C1, y1)
log_pi = ft.log_prior(keys)
U = upper_bounds(k_B, n_B, np.log(T['delta_1']) + log_pi)
eligible = U <= np.where(cls == 1, LOOSEST[1], LOOSEST[0])
el = np.flatnonzero(eligible)
t0 = time.time()
verdicts, times = verify_boxes(ft.grid, [(keys[i], int(cls[i])) for i in el])
z3res, _ = z3_sample(ft.grid, [b for b in verdicts if verdicts[b] == 'sound'], 60, SEED, timeout_ms=10000, time_budget_s=150)
VERIFY = {'eligible_boxes': int(len(el)), 'sound': sum(v == 'sound' for v in verdicts.values()),
          'unknown': sum(v == 'unknown' for v in verdicts.values()), 'z3_sample': z3res, 'seconds': time.time() - t0}
verified = np.full(len(keys), 'not needed', dtype=object)
verified[el] = [verdicts[(keys[i], int(cls[i]))] for i in el]
DF = pd.DataFrame({'key': [json.dumps(k) for k in keys], 'box_class': cls, 'size': [len(k) for k in keys],
                   'n_B': n_B, 'k_B': k_B, 'log_prior': log_pi, 'U': U, 'eligible': eligible, 'verified': verified})
f = CERT_DIR / f'{TASK}__boxes.parquet'
DF.to_parquet(f, index=False)
BOX_SHA = sha256_file(f)
U_rel = np.where(DF.verified.to_numpy() == 'sound', U, 1.0)       # unverified boxes are never released
print(f'calibration split 1: {len(cal1)} records ({int(y1.sum())} with the outcome); {len(keys)} boxes; second check {VERIFY}')

## Release, policy certification on calibration split 2 and test results

In [ ]:
box_index = {(k, int(cl)): i for i, (k, cl) in enumerate(zip(keys, cls))}
size = DF['size'].to_numpy()
parts = {}
for name, rr in (('cal2', cal2_rows), ('test', test_rows)):
    Cn = ft.grid.cells(Xa[rr]); yh = ft.pred.predict_cells(Cn).astype(int)
    pp, pb = pair_arrays([gen.cache[gen.cell_key(cc)] for cc in Cn], box_index, ft.M)
    parts[name] = {'rows': rr, 'yhat': yh, 'y': y[rr], 'pp': pp, 'pb': pb}
alpha_sel = {0: T['alpha_sel']['negative_outcome'], 1: T['alpha_sel']['enrichment']}
MAIN, POLICY, RELEASED = [], None, None
for grid, (a0s, betas) in GRIDS.items():
    for a0 in a0s:
        for beta in betas:
            a1 = round(1 - beta, 10)
            is_main = grid == 'scaled' and a0 == T['alpha_0'] and beta == T['beta']
            out = {n: release_patients(p['pp'], p['pb'], len(p['rows']), size, cls, n_B, U_rel, a0, a1) for n, p in parts.items()}
            pol = policy_test(out['cal2'], parts['cal2']['yhat'], parts['cal2']['y'], alpha_sel, T['delta_2'])
            if is_main:
                POLICY, RELEASED = pol, out['test']
            pt = parts['test']; ks = kind_summary(out['test'], pt['yhat'], pt['y'])
            for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
                rel = out['test']; a = (rel >= 0) & (pt['yhat'] == k); rb = rel[a]
                MAIN.append({'task': TASK, 'grid': grid, 'kind': kind, 'alpha_0': a0, 'beta': beta, 'target': a0 if k == 0 else a1,
                             'frozen_targets': is_main, 'cal2_accepted': pol[k]['accepted'], 'cal2_errors': pol[k]['errors'],
                             'U_sel': pol[k]['U_sel'], 'alpha_sel': pol[k]['alpha_sel'],
                             'policy_certified': pol[k]['certified'] if is_main else None,
                             'test_patients': len(pt['rows']), 'test_certified_share': ks[k]['share_of_patients'],
                             'test_certified_share_of_predicted_class': float(a.sum() / max(1, (pt['yhat'] == k).sum())),
                             'test_accepted': ks[k]['accepted'], 'test_errors': ks[k]['errors'], 'test_error_rate': ks[k]['error_rate'],
                             'test_error_lower_95': ks[k]['error_lower_95'], 'test_error_upper_95': ks[k]['error_upper_95'],
                             'mean_size': float(size[rb].mean()) if a.any() else None,
                             'median_calibration_support': float(np.median(n_B[rb])) if a.any() else None})
MAIN = pd.DataFrame(MAIN)
pd.set_option('display.width', 250)
print(MAIN[MAIN.frozen_targets].round(4).T.to_string())
for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
    p = POLICY[k]
    print(f"{kind}: U_sel {p['U_sel']:.4f} against alpha_sel {p['alpha_sel']} on {p['accepted']} accepted -> "
          f"{'certified' if p['certified'] else 'policy not certified'}")
for grid in GRIDS:
    print(f'\n{grid} grid, share of test patients with a certified explanation:')
    print(MAIN[MAIN.grid == grid].pivot_table(index=['kind', 'alpha_0'], columns='beta', values='test_certified_share').round(3).to_string())

## Released explanations, consistency check and the underlying model

In [ ]:
pt = parts['test']; rel = RELEASED
REL = pd.DataFrame([{'task': TASK, 'row': int(pt['rows'][i]), 'predicted': int(pt['yhat'][i]), 'released': int(rel[i]) >= 0,
                     'explanation': ft.grid.describe(ft.grid.from_key(keys[int(rel[i])])) if rel[i] >= 0 else None,
                     'size': int(size[int(rel[i])]) if rel[i] >= 0 else None, 'n_B': int(n_B[int(rel[i])]) if rel[i] >= 0 else None,
                     'U': float(U[int(rel[i])]) if rel[i] >= 0 else None} for i in range(len(rel))])
for k, kind in ((0, 'negative outcome'), (1, 'enrichment')):
    rr = REL[REL.released & (REL.predicted == k)]
    print(f'\n{kind}: most frequent released explanations'); print(rr.explanation.value_counts().head(8).to_string())
test_all = splits.indices(code, 'test')
Ct, yt = ft.grid.cells(Xa[test_all]), y[test_all]
ids = np.unique(rel[rel >= 0])
CONS = []
if len(ids):
    n_t, k_t = box_counts([keys[i] for i in ids], cls[ids], Ct, yt)
    for j, i in enumerate(ids):
        target = T['alpha_1'] if cls[i] == 1 else T['alpha_0']
        low = float(cp_lower(int(k_t[j]), int(n_t[j]), 0.05 / len(ids))) if n_t[j] else 0.0
        CONS.append({'task': TASK, 'box': DF.key.iat[i], 'box_class': int(cls[i]), 'size': int(size[i]), 'calibration_n': int(n_B[i]),
                     'calibration_k': int(k_B[i]), 'U': float(U[i]), 'target': target, 'test_n': int(n_t[j]), 'test_k': int(k_t[j]),
                     'test_error': float(k_t[j] / n_t[j]) if n_t[j] else None, 'test_lower_corrected': low,
                     'estimate_above_target': bool(low > target)})
CONS = pd.DataFrame(CONS)
if len(CONS):
    print(CONS.groupby('box_class').agg(released_boxes=('box', 'size'), median_test_n=('test_n', 'median'),
                                        mean_test_error=('test_error', 'mean'), flagged=('estimate_above_target', 'sum')).to_string())
margin = ft.pred.score(Xa[test_all])
METRICS = pd.DataFrame([{'task': TASK, **model_metrics(1 / (1 + np.exp(-margin)), yt, ft.pred.predict(Xa[test_all]))}])
print(METRICS.round(4).T.to_string())

## Figure: certified share against the target

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0))
# the negative-outcome share depends on alpha_0 only and the enrichment share on beta only
neg = MAIN[MAIN.kind == 'negative outcome'].drop_duplicates('alpha_0').sort_values('alpha_0')
enr = MAIN[MAIN.kind == 'enrichment'].drop_duplicates('beta').sort_values('beta')
for ax, d, x, scaled, lab, main in ((axes[0], neg, 'alpha_0', T['alpha_0_grid'], r'Target error for negative-outcome claims ($\alpha_0$)', T['alpha_0']),
                                    (axes[1], enr, 'beta', T['beta_grid'], r'Claimed minimum share with the outcome ($\beta$)', T['beta'])):
    on = d[x].isin(scaled)
    ax.plot(d[x], d.test_certified_share, color='#0072B2', lw=1.4, zorder=2)
    ax.plot(d[x][on], d.test_certified_share[on], 'o', ms=5, color='#0072B2', label='Scaled grid', zorder=3)
    ax.plot(d[x][~on], d.test_certified_share[~on], 'o', ms=5, mfc='white', mec='#0072B2', mew=1.2,
            label="Plan's grid (descriptive)", zorder=3)
    m = d[np.isclose(d[x], main)]
    ax.plot(m[x], m.test_certified_share, 'o', ms=10, mfc='none', mec='#D55E00', mew=1.5, label='Frozen target (policy test)', zorder=4)
    ax.set_xlabel(lab)
    ax.set_ylim(0, max(0.05, 1.15 * float(d.test_certified_share.max())))
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
axes[0].set_ylabel('Cardio-70k: share of test patients')
axes[0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[1].set_title('Certified enrichment explanations', fontsize=8.5)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=7, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.07, 1, 1))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_cardio_certified_share.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
MAIN.to_csv(f'{paths.RESULTS}/09d_main.csv', index=False)
REL.to_csv(f'{paths.RESULTS}/09d_test_releases.csv', index=False)
CONS.to_csv(f'{paths.RESULTS}/09d_box_consistency.csv', index=False)
METRICS.to_csv(f'{paths.RESULTS}/09d_model_metrics.csv', index=False)
write_json(f'{paths.RESULTS}/09d_results.json', {
    'frozen_config_sha256': ft.cfg_sha, 'box_table_sha256': BOX_SHA, 'limit': LIMIT, 'verification': VERIFY,
    'candidate_cells': len(gen.cache), 'median_seconds_per_cell': float(np.median(list(gen.seconds.values()))),
    'policy': {('negative_outcome' if k == 0 else 'enrichment'): v for k, v in POLICY.items()},
    'main': MAIN[MAIN.frozen_targets].to_dict('records'), 'model_metrics': METRICS.to_dict('records'),
    'consistency_flags': int(CONS.estimate_above_target.sum()) if len(CONS) else 0})
print('written: results/09d_main.csv, 09d_test_releases.csv, 09d_box_consistency.csv, 09d_model_metrics.csv, 09d_results.json; '
      'figures/fig_cardio_certified_share')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "09d: Cardio-70k certificates on calibration split 1, release, policy certification on calibration split 2, test results, consistency check, model metrics, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)